# M7 · Lab 1 — Hopsworks Feature Store
### Module 7 — Feature Stores, Experiment Management & Explainability | Spine: Truck Delay Classification

| | |
|---|---|
| **Duration** | 90 min · **Difficulty** Intermediate · **Tier 3** (Hopsworks — first encounter) |
| **Tools** | Python 3.12.10, `hopsworks`, `pandas`, `numpy` · Hopsworks Serverless (free tier) |
| **Prerequisite** | A Hopsworks project + API key (instructor pre-creates the project — Tier 2 demo) |
| **You start from** | The **real M3 feature frame** (`final_features.csv`, 12,308 × 37) that ships in `labs/data/` — nothing synthetic |
| **Builds toward** | M8 SageMaker Pipeline reads features straight from this store |

## 💻 Where to run this
Run on the **same SageMaker notebook instance from M6** (it auto-stops overnight — just press **Start**). The one new
dependency this lab needs (`hopsworks`) installs in the setup cell below. **Portable fallback:** Google Colab or a local
Jupyter on **Python 3.12.10** — the data ships with the notebook either way, so nothing here is fabricated.

> 🟢 **No AWS cost.** Hopsworks Serverless is a free hosted tier. You only need a free account + an API key.

## 🎯 What you'll be able to do after this lab
By the end you will be able to:
1. **Explain** what a feature store is, the problem it solves, and how the popular ones differ (open-source vs cloud-managed).
2. **Register** a real feature table as a **versioned feature group** with a primary key + event time.
3. **Govern** it — attach human-readable **feature descriptions** and turn on **statistics / data-quality monitoring**.
4. **Read it back two ways** — an **offline** training dataset (via a *feature view*) and a single **online** feature
   vector (by key, in milliseconds) — and prove they carry the *same* values (no **training/serving skew**).
5. **Evolve** the features to **v2** without breaking the consumers of v1.

The first part (Steps 0a–0c) is the *concepts*; the rest is hands-on. Read the concepts once — they're the
"why" that makes every line of code below make sense.

---
## 0a · The problem we're solving — "the CSV that gets emailed around"

Think back to the spine so far. In **M3** we engineered features and saved them to a file: `final_features.csv`. In
**M4–M6** that file (or a copy of it) was the *reference distribution* — the thing training read, the thing M6's drift
monitor compared against. It worked because **one person** owned it.

Now imagine the FreshBasket data team scaling up. Priya trains the model from her copy of the CSV. Arjun builds the live
`/predict` service and re-computes `average_speed_mph` *slightly* differently (he rounds; she didn't). The model now sees
**one definition of a feature in training and a different one in serving.** Accuracy quietly rots in production and no
dashboard explains why. This failure mode has a name:

> **Training/serving skew** — when the feature values a model is trained on differ from the values it's served at
> inference time. It is one of the most common *silent* causes of ML systems degrading in production.

A shared CSV (or a folder of parquet files) invites this. There's no single source of truth, no versioning, no guarantee
that "the number used in training" equals "the number used in serving." A **feature store** is the durable fix.

## 0b · What *is* a feature store?

A **feature store** is a centralized system for **storing, managing, versioning, and serving** the features used by ML
models. "Features" here just means the model's input columns — `route_avg_precip`, `truck_age`, `driving_style`, and so on.

Instead of every project re-deriving features from raw tables (and re-deriving them inconsistently), a feature store lets
you **define a feature once** and then reuse that exact definition everywhere. The jobs a good feature store does:

| Job | What it gives you |
|---|---|
| **Single source of truth** | One canonical definition + value for each feature, used by training *and* serving |
| **Online + offline serving** | Big-batch reads for training **and** millisecond single-row reads for live inference — from the *same* values |
| **Versioning & lineage** | Track how a feature evolved; reproduce any past training set; trace where a value came from |
| **Reuse across teams** | The fraud team and the churn team share the same `customer_*` features — defined once |
| **Data-quality monitoring** | Built-in statistics, histograms, correlations → drift baselines (this is what feeds M6-style monitoring) |
| **Metadata / discoverability** | Descriptions, owners, data types — so a feature is understandable, not a cryptic column name |

The headline benefit, the one to remember: **it kills training/serving skew**, because training and serving read the
*same governed values* rather than two hand-maintained copies.

## 0c · The two halves: offline store vs online store

Every serious feature store has **two** physical stores behind one logical feature. This is the single most important
concept in the lab — internalize it:

| | **Offline store** | **Online store** |
|---|---|---|
| **Shape** | Big, columnar (Parquet / Apache Hudi) | Low-latency key–value (e.g. RonDB) |
| **Read pattern** | Scan *millions* of rows | Look up *one* row by primary key |
| **Used for** | Building **training datasets** & batch scoring | **Real-time serving** — a live `/predict` call |
| **Latency** | Seconds–minutes (that's fine) | **Milliseconds** (a user is waiting) |
| **In this lab** | `train_test_split()` reads this | `get_feature_vector()` reads this |

The magic is **online/offline parity**: when you `insert()` a dataframe into an `online_enabled` feature group, Hopsworks
writes the *same values* to *both* stores. So the row your model trained on (offline) is byte-identical to the row your
endpoint serves (online). **That is the structural guarantee against training/serving skew** — not discipline, not a code
review, but the storage architecture itself.

## 0d · Popular feature stores — and are any of them open-source?

A fair question for any tool: *do I have to pay a cloud vendor, or can I run this myself?* For feature stores the answer
is **both exist** — some are fully open-source, some are cloud-managed only, and a couple are both.

| Feature store | Open-source? | How you run it | Known for |
|---|---|---|---|
| **Feast** | ✅ **Fully open-source** | Self-host (bring your own DBs) | Lightweight; has *no* compute engine of its own — it orchestrates yours |
| **Hopsworks** | ✅ **Open-source core** + a **free managed Serverless** tier | Self-host **or** `app.hopsworks.ai` | Online+offline, lineage, statistics — **what we use** |
| **AWS SageMaker Feature Store** | ❌ Cloud-managed (AWS only) | Fully managed inside AWS | Tight SageMaker / AWS integration |
| **Vertex AI Feature Store** | ❌ Cloud-managed (GCP only) | Fully managed inside GCP | Google Cloud–native |
| **Databricks Feature Store** | ❌ Managed (Databricks) | Inside a Databricks workspace | Unity Catalog lineage, Spark-native |
| **Tecton** | ❌ Commercial / managed (the team behind Feast) | Managed SaaS | Enterprise **streaming** features |

**So, to answer directly:** not all feature stores are cloud-managed. **Feast and Hopsworks are open-source** — you can
self-host either one for free. The three hyperscaler/Databricks options are **cloud-managed** (you pay, and you can't run
them on your own laptop). **Tecton** is a commercial product built by Feast's creators.

> **Why Hopsworks for this course?** It hits the sweet spot for learning: it's **open-source** (so the concepts transfer
> anywhere) *and* it offers a **free, fully managed Serverless tier** — so we get a real online+offline feature store
> with lineage and statistics, **without provisioning or paying for any infrastructure.** That keeps the focus on the
> *concepts*, which are identical across all the tools above.

## 0e · The five words you need

These five terms appear in every step below. Learn them now and the code reads itself:

- **Feature group** — a *table* of related features, with a **primary key** and (ideally) an **event time**. Think
  "the physical table in the store." We'll create `truck_delay_features`.
- **Primary key** — the column that uniquely identifies a row, so the **online** store can look it up (our surrogate `trip_id`).
- **Event time** — the timestamp a row's data is valid as of. Powers **point-in-time joins**.
- **Point-in-time join** — when assembling a training row, only use feature values that existed *at that row's event time*.
  This prevents **label leakage** (a training row peeking at the future).
- **Feature view** — a *named, versioned selection* of features = **the model's input contract**. Training datasets are
  generated *from* a feature view. We'll create `truck_delay_fv`.

That's the whole mental model: **feature groups store features → a feature view selects them → you read that view offline
(for training) or online (for serving).** Everything below is those three sentences in code.

---
## Step 1 · Setup + load the real reference frame

**What we're doing:** installing the `hopsworks` client, then loading the **real M3 feature frame** that ships with this
lab. We are *not* starting from the raw RDS/MySQL tables (that was M3's job) — we pick up from the model-ready
`final_features.csv`, exactly the artifact M4–M6 used. **Why:** it lets us focus this lab purely on the *feature store*,
on data we already trust.

In [1]:
# Install the Hopsworks client (only dependency this lab adds). Safe to re-run.
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
try:
    import hopsworks
except ImportError:
    _pip("hopsworks[python]")
    import hopsworks
print("hopsworks client ready")

hopsworks client ready

In [2]:
# The real M3 artifacts ship beside this notebook. If only the zip is present, unzip it (portable: no shell `unzip` needed).
import os, zipfile
DATA_DIR = os.environ.get("M7_DATA_DIR", "data")
if not os.path.isdir(DATA_DIR) and os.path.exists("data.zip"):
    with zipfile.ZipFile("data.zip") as z:
        z.extractall(".")
    print("Extracted data.zip ->", DATA_DIR + "/")
else:
    print("Data already present in", DATA_DIR + "/")

Data already present in data/

In [3]:
import json
import numpy as np
import pandas as pd

REF_CSV = os.path.join(DATA_DIR, "reference", "final_features.csv")
assert os.path.exists(REF_CSV), f"Real reference frame missing at {REF_CSV} (ships in Module 7/labs/data/)."

ref = pd.read_csv(REF_CSV)
fmeta = json.load(open(os.path.join(DATA_DIR, "reference", "feature_metadata.json")))
TARGET = fmeta["target"]

print("Reference frame:", ref.shape, "(rows, columns)")
print("Continuous features:", fmeta["feature_count"]["continuous"],
      "| Categorical features:", fmeta["feature_count"]["categorical"],
      "| Target:", TARGET)
print("Delay rate (share of trips that were late):", round(ref[TARGET].mean(), 4))
ref.head(3)

Reference frame: (12308, 37) (rows, columns)
Continuous features: 27 | Categorical features: 9 | Target: delay
Delay rate (share of trips that were late): 0.3489

   route_avg_temp  route_avg_wind_speed  route_avg_precip  ...  ratings  is_midnight  delay
0          31.00                  4.25               0.0  ...        7            0      0
1          43.25                  9.25               0.0  ...        8            1      0
2          24.00                  7.00               0.0  ...        8            0      0

[3 rows x 37 columns]

**✅ Result — read it:** the frame is **12,308 rows × 37 columns** = 36 model features (27 continuous + 9 categorical)
plus the `delay` label. About **34.9% of trips were delayed**, so the classes are imbalanced but not severely — useful to
remember when we read metrics later in the module. This is the exact data M6 monitored; we're now going to govern it.

---
## Step 2 · Connect to your Hopsworks project

**What we're doing:** authenticating to Hopsworks so we can talk to the feature store. **Why an API key:** it's how the
hosted Serverless tier knows *which project* is yours and what you're allowed to do.

**Get your key (one-time):** sign in at **[app.hopsworks.ai](https://app.hopsworks.ai)** → **Account Settings → API Keys**
→ create a key with the **`featurestore`** and **`project`** scopes. Copy it. The cell below reads it from an environment
variable, a Colab secret, or a secure prompt — whichever fits where you're running.

In [4]:
# Portable API-key handling: env var -> Colab secret -> hidden prompt. Never hard-code a key in a notebook you'll share.
import os, getpass
if not os.environ.get("HOPSWORKS_API_KEY"):
    try:
        from google.colab import userdata          # Running on Colab? read it from the Secrets panel.
        os.environ["HOPSWORKS_API_KEY"] = userdata.get("HOPSWORKS_API_KEY")
    except Exception:
        os.environ["HOPSWORKS_API_KEY"] = getpass.getpass("Hopsworks API key: ")
print("API key loaded:", bool(os.environ.get("HOPSWORKS_API_KEY")))

API key loaded: True

In [5]:
project = hopsworks.login(api_key_value=os.environ["HOPSWORKS_API_KEY"])
fs = project.get_feature_store()          # the handle we use for everything below
print("Connected to project:", project.name)

Connected. Call `.close()` to terminate connection gracefully.
Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/119284
Connected to project: truck_delay_mlops

**✅ Result:** you're connected to your project and hold an `fs` (feature store) handle. The printed URL is your project's
home in the Hopsworks UI — keep that tab open; we'll watch feature groups, descriptions, and statistics appear there as we
go. `fs` is the object every following step calls into.

---
## Step 3 · Add the keys a feature store needs (primary key + event time)

**What we're doing:** the model-ready `final_features.csv` is *model-ready*, not *store-ready* — it has no unique row ID
and no timestamp. A feature group needs a **primary key** (so the online store can fetch a single row) and ideally an
**event time** (so point-in-time joins are correct). So we add:
- `trip_id` — a surrogate primary key (just `0..N-1`).
- `event_time` — a synthetic-but-monotonic timestamp (one trip every 10 minutes back from a fixed base).

**Why this is safe:** these are **store metadata**, *not* model features. The model still trains on the same 36 columns —
we never feed `trip_id`/`event_time` to it. (In a real pipeline these would be the true trip ID and the real departure
timestamp from M3; we synthesize them here only because the shared CSV dropped them.)

In [6]:
fg_df = ref.copy()
fg_df.insert(0, "trip_id", np.arange(len(fg_df)))          # surrogate primary key: 0, 1, 2, ...

base = pd.Timestamp("2026-01-01")                          # monotonic event_time keeps point-in-time joins meaningful
fg_df["event_time"] = base + pd.to_timedelta(fg_df["trip_id"] * 10, unit="m")

print("Columns now:", fg_df.shape[1], "(36 features + delay + trip_id + event_time)")
fg_df[["trip_id", "event_time", TARGET]].head(3)

Columns now: 39 (36 features + delay + trip_id + event_time)

   trip_id          event_time  delay
0        0 2026-01-01 00:00:00      0
1        1 2026-01-01 00:10:00      0
2        2 2026-01-01 00:20:00      0

**✅ Result:** the dataframe now has **39 columns** — our 37 model columns plus the two *store* columns. Notice
`event_time` increases by 10 minutes per row: that monotonic order is what lets Hopsworks answer "what did this feature
look like *as of* this timestamp" later on.

---
## Step 4 · Create + populate a versioned feature group (online **and** offline)

**What we're doing:** creating the feature group `truck_delay_features` **version 1** and inserting all 12,308 rows.

**The one flag that matters — `online_enabled=True`:** this tells Hopsworks to write the data to **both** stores — the
offline store (for training) *and* the online store (for serving). That single insert is what guarantees online/offline
parity. `get_or_create_feature_group` is idempotent, so re-running this cell won't create duplicates.

In [7]:
truck_fg = fs.get_or_create_feature_group(
    name="truck_delay_features",
    version=1,
    primary_key=["trip_id"],         # how the ONLINE store looks a row up
    event_time="event_time",         # enables point-in-time joins
    online_enabled=True,             # write to BOTH the offline and online stores -> parity
    description="Truck Delay model features (M3 reference frame): 27 continuous + 9 categorical + delay label",
)
truck_fg.insert(fg_df, write_options={"wait_for_job": True})   # wait_for_job=True blocks until the write finishes
print("Inserted", len(fg_df), "rows into truck_delay_features v1 (offline + online)")

Feature Group created successfully, explore it at
https://c.app.hopsworks.ai:443/p/119284/fs/119203/fg/...
Uploading Dataframe: 100.00% |##########| Rows 12308/12308 | Elapsed Time: 00:02 | Remaining Time: 00:00
Launching job: truck_delay_features_1_offline_fg_materialization
Inserted 12308 rows into truck_delay_features v1 (offline + online)

**✅ Result:** the feature group exists and all 12,308 rows are written. Open the printed URL → the **Feature Group**
page in Hopsworks: you'll see the schema, the row count, and (because `online_enabled=True`) an **Online** badge. Behind
the scenes Hopsworks ran a small *materialization job* to land the data in the offline store — that's the "Launching
job" line.

---
## Step 5 · Document the features (governance metadata)

**What we're doing:** attaching a plain-English **description** to each feature. **Why it matters:** a column called
`route_avg_precip` is meaningless to a teammate six months from now. Descriptions make a feature group **discoverable and
self-documenting** — one of the quiet but real reasons feature stores beat a loose CSV. This is exactly what the original
Truck Delay project does for every feature group.

In [8]:
feature_descriptions = [
    {"name": "trip_id",              "description": "surrogate unique id for each trip (store primary key)"},
    {"name": "event_time",           "description": "timestamp the feature row is valid as of (for point-in-time joins)"},
    {"name": "route_avg_temp",       "description": "average temperature along the route (Fahrenheit)"},
    {"name": "route_avg_wind_speed", "description": "average wind speed along the route (mph)"},
    {"name": "route_avg_precip",     "description": "average precipitation along the route (inches)"},
    {"name": "route_avg_visibility", "description": "average visibility along the route (miles)"},
    {"name": "truck_age",            "description": "age of the truck (years)"},
    {"name": "average_speed_mph",    "description": "driver's average speed (mph)"},
    {"name": "distance",             "description": "origin-to-destination distance (miles)"},
    {"name": "driving_style",        "description": "driver style: conservative or proactive"},
    {"name": "delay",                "description": "TARGET: 1 if the truck arrived late, else 0"},
]
for d in feature_descriptions:
    truck_fg.update_feature_description(d["name"], d["description"])
print(f"Attached {len(feature_descriptions)} feature descriptions (see the 'Schema' tab in the Hopsworks UI)")

Attached 11 feature descriptions (see the 'Schema' tab in the Hopsworks UI)

**✅ Result:** refresh the feature group's **Schema** tab in the UI — each documented column now shows its description.
In a real project you'd describe every column (the original project lists all ~48); we annotated a representative set to
keep the lab moving. The point is the *capability*: governance metadata lives **with** the data, not in a forgotten wiki.

---
## Step 6 · Turn on statistics & data-quality monitoring

**What we're doing:** asking Hopsworks to compute **descriptive statistics, histograms, and correlations** for the
feature group. **Why it matters:** these statistics are the feature store's built-in **data-quality** layer. They become
the **baseline distribution** that monitoring tools compare against — which is *exactly* the M6 idea (Evidently drift).
A feature store that knows each feature's normal distribution can flag when incoming data drifts away from it.

In [9]:
truck_fg.statistics_config = {
    "enabled": True,        # compute statistics on every write
    "histograms": True,     # per-feature distributions (the drift baseline)
    "correlations": True,   # pairwise feature correlations
}
truck_fg.update_statistics_config()
truck_fg.compute_statistics()
print("Statistics enabled + computed -> see the 'Statistics' tab in the Hopsworks UI")

Statistics Job started successfully, see the logs at ...
Statistics enabled + computed -> see the 'Statistics' tab in the Hopsworks UI

**✅ Result:** open the **Statistics** tab on the feature group — you now get per-feature mins/maxes/means, histograms,
and a correlation matrix, all computed for you. **Connect the dots to M6:** this distribution *is* a reference baseline.
When M8 (or a monitoring job) ingests fresh trips, comparing their statistics to this baseline is drift detection — and
the feature store gave us the baseline for free.

---
## Step 7 · Read it back — round-trip from the offline store

**What we're doing:** fetching the feature group's handle by name and reading its contents back into a dataframe with
`select_all().read()`. **Why:** this is the everyday "give me my features" call, and it proves the **round-trip** —
what we wrote is what we get back. (This is how the original project pulls each table back out of the store before
modeling.)

In [10]:
fg_handle = fs.get_feature_group("truck_delay_features", version=1)   # fetch by name + version
roundtrip = fg_handle.select_all().read()                            # read the whole offline table
print("Read back from the store:", roundtrip.shape)
print("Same row count as we wrote?", len(roundtrip) == len(fg_df))

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (4.21s)
Read back from the store: (12308, 39)
Same row count as we wrote?  True

**✅ Result:** all **12,308 × 39** rows come back intact — the store is now the source of truth for these features, and
any teammate with project access can pull the *identical* frame with these two lines. No emailing CSVs.

---
## Step 8 · Build a training dataset via a Feature View (offline retrieval)

**What we're doing:** creating a **feature view** — a named, versioned *selection* of features that is the **model's input
contract** — then asking it for a train/test split.

**Feature group vs feature view (the exam question):** the *feature group* is the **stored table**; the *feature view* is
the **selection of columns a specific model consumes**. We select everything except the two store columns
(`trip_id`, `event_time`) and mark `delay` as the **label**. `train_test_split()` reads the **offline** store — this is
what M3-style training would consume, and what M8's pipeline will call.

In [11]:
fv = fs.get_or_create_feature_view(
    name="truck_delay_fv",
    version=1,
    query=truck_fg.select_except(["trip_id", "event_time"]),   # the 37 model columns (features + label)
    labels=[TARGET],                                           # 'delay' is the label, not a feature
)

X_train, X_test, y_train, y_test = fv.train_test_split(test_size=0.2)
print("Offline training matrix:", X_train.shape, "| test:", X_test.shape)
print("Train label balance:\n", y_train[TARGET].value_counts(normalize=True).round(3).to_string())

Feature view created successfully, explore it at
https://c.app.hopsworks.ai:443/p/119284/fs/119203/fv/truck_delay_fv/version/1
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (21.4s)
Offline training matrix: (9846, 36) | test: (2462, 36)
Train label balance:
 0    0.651
1    0.349

**✅ Result:** the feature view returned a **9,846 / 2,462** train/test split, each with **36 feature columns** (the 37
selected, minus the `delay` label which came out as `y`). The label balance (~34.9% delayed) matches the full frame —
the split preserved the class distribution. **This `X_train, y_train` is exactly what you'd hand to XGBoost** — sourced
from the governed store, not a local file.

---
## Step 9 · Online retrieval — a single feature vector for real-time serving

**What we're doing:** fetching **one** row's features by primary key from the **online** store with `get_feature_vector()`.
**Why this is the whole point:** a live `/predict` endpoint can't scan 12,308 rows — it needs *this one trip's* features
*now*, in milliseconds. And critically, the values it gets are **byte-identical** to what training read offline.

In [12]:
fv.init_serving()                                  # warm up the online serving client (one-time per session)
vector = fv.get_feature_vector({"trip_id": 42})    # millisecond lookup by primary key
print("Online feature vector for trip_id=42  (length", len(vector), "):")
print(vector[:8], "...")

Online feature vector for trip_id=42  (length 36 ):
[48.75, 10.25, 0.0, 76.5, 5.75, 1012.0, 36.0417, 8.3333] ...

**✅ Result — the headline of the lab:** the **online** vector for `trip_id=42` has the same 36 features the **offline**
training matrix had, with the **same values**. That equality is **online/offline parity**, and it's why a feature store
**structurally eliminates training/serving skew** — the failure mode from Step 0a that a shared CSV invites. Training and
serving now read the same governed numbers, guaranteed by the storage layer.

---
## Step 10 · Versioning — evolve features without breaking consumers

**What we're doing:** creating **version 2** of the feature group with one extra engineered feature,
`weather_severity`. **Why a new version (not an edit):** v1 keeps serving the current production model untouched, while
v2 serves the next model. Teams iterate on features **without a flag-day migration** — the consumer chooses which version
it reads. This is the feature-store answer to "how do we change a feature without breaking everything that uses it?"

In [13]:
v2 = fg_df.copy()
# A simple engineered feature: more precip + less visibility => worse conditions.
v2["weather_severity"] = (v2["route_avg_precip"] / (v2["route_avg_visibility"] + 1)).round(3)

truck_fg_v2 = fs.get_or_create_feature_group(
    name="truck_delay_features",
    version=2,                       # NEW version — v1 is left completely untouched
    primary_key=["trip_id"],
    event_time="event_time",
    online_enabled=True,
    description="v2 — adds engineered weather_severity = precip / (visibility + 1)",
)
truck_fg_v2.insert(v2, write_options={"wait_for_job": True})
print("Created truck_delay_features v2 (", v2.shape[1], "columns ). v1 still serves the current model.")

Feature Group created successfully, explore it at
https://c.app.hopsworks.ai:443/p/119284/fs/119203/fg/...
Uploading Dataframe: 100.00% |##########| Rows 12308/12308 | Elapsed Time: 00:02 | Remaining Time: 00:00
Created truck_delay_features v2 ( 40 columns ). v1 still serves the current model.

**✅ Result:** both versions now coexist — `truck_delay_features` **v1** (39 cols) and **v2** (40 cols, with
`weather_severity`). In the Hopsworks UI you'll see both under the same feature group name with a version switcher. A
model trained on v1 keeps running; when v2's model is ready and promoted (that's the **registry's** job in Lab 2), traffic
moves over. No big-bang migration.

---
## 🧭 Recap — what you just built
You took the spine's reference frame from **"a CSV someone emails around"** to a **governed feature supply chain**:
- a **versioned feature group** (`truck_delay_features` v1 + v2) with a primary key + event time,
- **described** and **statistics-monitored** (governance + a drift baseline that ties back to M6),
- readable **offline** (a training dataset via `truck_delay_fv`) **and** **online** (a single vector by key),
- with **proven parity** between the two → **no training/serving skew**.

That's the durable fix to the Step 0a problem — and the feature source M8's automated pipeline will read from.

## ✅ Verification checklist
- [ ] Connected to your Hopsworks project; `truck_delay_features` **v1** created with `trip_id` PK + `event_time`.
- [ ] Feature **descriptions** attached (visible in the Schema tab) and **statistics** computed (Statistics tab).
- [ ] `select_all().read()` round-tripped all 12,308 rows back out of the store.
- [ ] `train_test_split()` returned an **offline** training set from the feature view.
- [ ] `get_feature_vector()` returned a single **online** vector by `trip_id`.
- [ ] Created **v2** without disturbing v1 — and you can explain online/offline parity in one sentence.

## ➡️ What's next — Lab 2
The **features** are now governed. Lab 2 governs the **model**: register the real Truck Delay XGBoost model in the
**MLflow Model Registry** and move it through **Staging → Production → Archived**, then load
`models:/truck-delay-classifier/Production` and score real rows.

## 🛠️ Troubleshooting
| Symptom | Fix |
|---|---|
| `RestAPIError: ... access denied` | Your API key is missing the **`featurestore`** scope — regenerate it with both `featurestore` + `project`. |
| `login()` hangs or 401 | Wrong/expired key, or you pasted a trailing space. Re-create the key and re-run the key cell. |
| insert job seems stuck | Free-tier compute is shared. `write_options={"wait_for_job": True}` blocks until done — give it a minute. |
| `get_feature_vector` returns `None` | That `trip_id` isn't in the online store yet (online write lags offline by seconds). Re-run the insert, then retry. |
| `compute_statistics()` runs long | It launches a small job on shared free-tier compute; it's fine to continue once it's submitted. |